# 🧠 TALOS — Reindex de Embeddings de Alto Rendimiento con Qwen3-Embedding-0.6B

**Modelo:** `Qwen/Qwen3-Embedding-0.6B` (600M Parámetros — SOTA Retrieval Multilingüe)
**Dimensiones:** `1024` (Float16 Nativo)
**Tiempo estimado en GPU T4:** ~20 a 30 minutos (287.000 chunks)
**Optimizaciones:** Checkpoints automáticos persistentes en Google Drive y Batch Size optimizado (128).

In [ ]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive montado correctamente")

In [ ]:
# 2. Limpieza de librerías en conflicto e instalación de dependencias
# Colab tiene torchvision y torchaudio preinstalados con versiones de CUDA que chocan con PyTorch.
# Al ser un pipeline de texto (NLP), los removemos para evitar errores de C++/CUDA.
!rm -rf /usr/local/lib/python*/dist-packages/~*
!pip uninstall -y -q torchvision torchaudio
!pip install -q -U sentence-transformers accelerate jedi


In [ ]:
# 3. Autenticación en Hugging Face (usando tu secreto HF_TOKEN de Colab)
from huggingface_hub import login
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        login(token=token)
        print("✅ Autenticado exitosamente en Hugging Face con tu HF_TOKEN.")
    else:
        print("ℹ️ No se detectó HF_TOKEN en Secrets (ícono 🔑 de la izquierda). Continuando con acceso libre sin token.")
except Exception as e:
    print(f"ℹ️ Acceso libre sin token: {e}")


In [ ]:
# 4. Verificación y optimización de memoria GPU
import os, torch, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU no detectada. En el menú superior ve a: Entorno de ejecución > Cambiar tipo de entorno de ejecución > Selecciona GPU T4")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU activa: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
DEVICE = "cuda"


In [ ]:
# 5. Cargar modelo Qwen/Qwen3-Embedding-0.6B en Float16 nativo
from sentence_transformers import SentenceTransformer
import torch, gc

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
DIMENSIONS = 1024
MAX_SEQ_LENGTH = 512   # Contrato TALOS: 512 tokens
BATCH_SIZE = 128       # Tamaño óptimo para 0.6B en GPU T4 (utilización máxima y segura)

gc.collect()
torch.cuda.empty_cache()

print(f"Cargando {MODEL_NAME} en Float16 nativo...")
model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.float16}
)
model.max_seq_length = MAX_SEQ_LENGTH

def embed_passages(texts, current_bs=BATCH_SIZE):
    with torch.no_grad():
        try:
            embeddings = model.encode(
                texts,
                batch_size=current_bs,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return embeddings.tolist() if hasattr(embeddings, "tolist") else [list(e) for e in embeddings]
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if current_bs > 16:
                half = len(texts) // 2
                return embed_passages(texts[:half], current_bs=current_bs // 2) + embed_passages(texts[half:], current_bs=current_bs // 2)
            raise

# Smoke test rápido
test_vec = embed_passages(["Prueba de verificación semántica TALOS"])[0]
assert len(test_vec) == DIMENSIONS, f"Error: {len(test_vec)} != {DIMENSIONS}"
vram_used = torch.cuda.memory_allocated() / (1024**3)
print(f"✅ Modelo listo: {DIMENSIONS} dimensiones en FP16 | VRAM en uso: {vram_used:.2f} GB.")


In [ ]:
# 6. Ejecutar vectorización del corpus con auto-reanudación (Checkpoints)
import os, glob, json, time, shutil, gc, torch
from tqdm.notebook import tqdm

DRIVE_FOLDER = "/content/drive/MyDrive/TALOS"
assert os.path.exists(DRIVE_FOLDER), f"No se encontró la carpeta {DRIVE_FOLDER} en tu Google Drive."

gc.collect()
torch.cuda.empty_cache()

# Buscar archivo pending más reciente
pending_files = sorted(glob.glob(f"{DRIVE_FOLDER}/pending_*.jsonl"))
if not pending_files:
    raise FileNotFoundError(f"No se encontró ningún archivo pending_*.jsonl en {DRIVE_FOLDER}")

latest_pending = pending_files[-1]
print(f"📄 Leyendo archivo de pendientes: {latest_pending}")

records = []
with open(latest_pending, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        obj = json.loads(line)
        if "_meta" in obj: continue
        records.append(obj)

print(f"📊 Total de chunks a vectorizar: {len(records):,}")

# Archivo de progreso persistente en Drive
checkpoint_file = f"{DRIVE_FOLDER}/vectors_checkpoint_qwen3_06b.jsonl"
already_done = set()
if os.path.exists(checkpoint_file):
    print(f"⚡ Detectado checkpoint previo. Analizando registros completados...")
    with open(checkpoint_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            o = json.loads(line)
            if "_meta" in o: continue
            already_done.add((o["chunk_id"], o["notebook_id"]))
    print(f"✅ Se reanudará desde el chunk {len(already_done):,}. Chunks ya procesados: {len(already_done):,}")

pending_to_process = [r for r in records if (r["chunk_id"], r["notebook_id"]) not in already_done]
print(f"⏳ Pendientes en esta ejecución: {len(pending_to_process):,}")

# Escribir encabezado de contrato si el archivo es nuevo
if len(already_done) == 0:
    with open(checkpoint_file, "w", encoding="utf-8") as out_fh:
        meta = {
            "_meta": {
                "schema": "talos-vectors-v1",
                "contract": {
                    "schema": "talos-embedding-contract-v1",
                    "model": MODEL_NAME,
                    "model_id": "Qwen3-Embedding-0.6B",
                    "provider_id": "sentence_transformers",
                    "dimensions": DIMENSIONS,
                    "max_length": 512,
                    "normalize_l2": True,
                    "pooling": "mean",
                    "prefixes": {"query": "", "passage": ""}
                }
            }
        }
        out_fh.write(json.dumps(meta) + "
")

# Procesamiento por lotes con guardado continuo
with open(checkpoint_file, "a", encoding="utf-8") as out_fh:
    for i in tqdm(range(0, len(pending_to_process), BATCH_SIZE), desc="Vectorizando corpus Qwen3-0.6B"):
        batch = pending_to_process[i:i+BATCH_SIZE]
        texts = [r.get("text", "") for r in batch]
        vecs = embed_passages(texts)
        for r, vec in zip(batch, vecs):
            out_fh.write(json.dumps({
                "chunk_id": r["chunk_id"],
                "notebook_id": r["notebook_id"],
                "vector": vec
            }) + "
")
        if (i // BATCH_SIZE) % 10 == 0:
            out_fh.flush()
    out_fh.flush()

# Al finalizar completamente, generar el archivo vectors_*.jsonl definitivo
final_vectors_file = f"{DRIVE_FOLDER}/vectors_{time.strftime('%Y%m%dT%H%M%SZ')}.jsonl"
shutil.copy2(checkpoint_file, final_vectors_file)
print(f"
🎉 ¡VECTORIZACIÓN COMPLETA!")
print(f"Archivo final listo para TALOS: {final_vectors_file}")